## Required preprocessing steps
- generate label file and change it to have a label for scrambled sequences in the design
- code for outlier removal 
- Update from max in April 2025: normalize the counts first using mpralib 
- Optional: harmonize the design headers

In [8]:
import pandas as pd
import os

# load helpful functions
import sys
sys.path.append('../../00_helpful_functions')
import helpful_functions as hf

In [9]:
input = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/20241113_80K_MPRAsnakeflow/results/experiments/mpra80KNeuronbbmapmapq30BC10DNA1RNA1/assigned_counts/assignmentFixDuplicates/default/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz"
input = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/20241113_80K_MPRAsnakeflow/results_design_before_metadata_file/experiments/mpra80KNeuronbbmapmapq30BC10DNA1RNA1/assigned_counts/assignmentFixDuplicates/bc10DNA1RNA1OutlierNo/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz"
input = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/20250214_80k_MPRA_bwa_finest/results/experiments/mpra80KNeuronbwaFinestRNA1/assigned_counts/assignmentFixDuplicates/default/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz"
input = "/data/cephfs-1/scratch/groups/kircher/MPRA/IGVF_submission/80K/mprasnakeflow/results/experiments/defaultWTC11/assigned_counts/defaultAssignment/default/WTC11_allreps_merged_barcode_assigned_counts.tsv.gz"
input = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/strand_sensitive_80K_NGN2/results/experiments/mpra80KNGN2bbmapmapq30BC10DNA1RNA1/assigned_counts/assignmentFixDuplicates/default/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz"

group_name_prefix = "bwa_finest_"
group_name_prefix = "strand_sensitive_" # remember "_" at the end of the name
group_name_prefix = "BBmap_"

cell_type_name = "WTC11_"
cell_type_name = "NGN2_"

# output paths
output_path = f"/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/{group_name_prefix}{cell_type_name}allreps_merged_barcode_assigned_counts.tsv.gz"
output_path_outliered = f"/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/{group_name_prefix}{cell_type_name}allreps_merged_barcode_assigned_counts_outlier_removed.tsv.gz"
output_path_split_hashtag = f"/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/{group_name_prefix}{cell_type_name}allreps_merged_barcode_assigned_counts_hashtag_expanded.tsv.gz"

print(output_path)
print(output_path_outliered)
print(output_path_split_hashtag)


bc_thresh = 10
BCalm_variant_input_df = pd.read_csv(input, sep="\t")

/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/BBmap_NGN2_allreps_merged_barcode_assigned_counts.tsv.gz
/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/BBmap_NGN2_allreps_merged_barcode_assigned_counts_outlier_removed.tsv.gz
/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/BBmap_NGN2_allreps_merged_barcode_assigned_counts_hashtag_expanded.tsv.gz


FileNotFoundError: [Errno 2] No such file or directory: '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/strand_sensitive_80K_NGN2/results/experiments/mpra80KNGN2bbmapmapq30BC10DNA1RNA1/assigned_counts/assignmentFixDuplicates/default/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz'

In [ ]:
print(BCalm_variant_input_df.shape[0]) # official strand sensitive: 6307700; bwa finest: 6402318 WTC11: 5031012
BCalm_variant_input_df.head()

5031012


,barcode,oligo_name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,13.0,4.0,13.0,NaN,NaN
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,9.0,3.0,1.0,15.0,2.0,21.0
2,GAGCACGACAACAGA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,5.0,11.0
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,10.0,15.0,2.0,19.0,NaN,NaN
4,CTCAGCGCCCCAATT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,2.0,5.0,NaN,NaN


In [4]:
# rename
column_names = ["Barcode", "name", "dna_count_1", "rna_count_1", "dna_count_2", "rna_count_2", "dna_count_3", "rna_count_3"]
BCalm_variant_input_df.columns = column_names


In [5]:
BCalm_variant_input_df.head()

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,13.0,4.0,13.0,NaN,NaN
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,9.0,3.0,1.0,15.0,2.0,21.0
2,GAGCACGACAACAGA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,5.0,11.0
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,10.0,15.0,2.0,19.0,NaN,NaN
4,CTCAGCGCCCCAATT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,2.0,5.0,NaN,NaN


In [ ]:
BCalm_variant_input_df['name'].nunique() # official strand sensitive: 76305; bwa finest: 77999; WTC11: 77715

77715

### Outlier removal for each replicate: 
- for 80K (selfmade strand sensitive): ~300K barcodes removed (<4.7% excluded because of outliers) => around 549 sequences missing
- for 80K (official strand sensitive option): 

In [7]:
print(f"lost barcodes (bc outlier removal): ~300k {round(300000/BCalm_variant_input_df.shape[0]*100, 2)}%") # percentage of lost barcodes because of outlier removal

lost barcodes (bc outlier removal): ~300k 4.76%


In [8]:
print(f'lost oligos (bc outlier removal): 549 {round(549/73288*100,2)}%')

lost oligos (bc outlier removal): 549 0.75%


In [7]:
def outlier_removal_by_rna_zscore(df, times_zscore = 3, rna_count_col='rna_count', col_name='name', col_barcode='Barcode'):
    df["mean"] = df.groupby(col_name)[rna_count_col].transform('mean')

    df["std"] = df.groupby(col_name)[rna_count_col].transform('std')
    df["rna_z_scores"] = (df[rna_count_col] - df["mean"]) / df["std"]

    m = df.rna_z_scores.abs() <= times_zscore
    barcodes_removed = df[~m][col_barcode]
    df = df[m]
    return df[m], barcodes_removed

In [8]:
# split into replicate information
rep1_df = BCalm_variant_input_df[['Barcode', 'name', 'dna_count_1', 'rna_count_1']].copy()
rep1_df = rep1_df.rename(columns={'dna_count_1':'dna_count', 'rna_count_1': 'rna_count'})
rep1_df.head()

,Barcode,name,dna_count,rna_count
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,13.0
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,9.0,3.0
2,GAGCACGACAACAGA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,10.0,15.0
4,CTCAGCGCCCCAATT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN


In [9]:
rep2_df = BCalm_variant_input_df[['Barcode', 'name', 'dna_count_2', 'rna_count_2']].copy()
rep2_df = rep2_df.rename(columns={'dna_count_2':'dna_count', 'rna_count_2': 'rna_count'})
rep2_df.head()
rep3_df = BCalm_variant_input_df[['Barcode', 'name', 'dna_count_3', 'rna_count_3']].copy()
rep3_df = rep3_df.rename(columns={'dna_count_3':'dna_count', 'rna_count_3': 'rna_count'})
rep3_df.head()

,Barcode,name,dna_count,rna_count
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2.0,21.0
2,GAGCACGACAACAGA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,5.0,11.0
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN
4,CTCAGCGCCCCAATT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN


In [10]:
## remove outliers with the given function
rep1_df_outliered, barcodes_removed_rep1 = outlier_removal_by_rna_zscore(rep1_df, times_zscore=3, col_barcode='Barcode')


/tmp/ipykernel_2908/3382132649.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  return df[m], barcodes_removed


In [11]:
rep2_df_outliered, barcodes_removed_rep2 = outlier_removal_by_rna_zscore(rep2_df, times_zscore=3)
rep3_df_outliered, barcodes_removed_rep3 = outlier_removal_by_rna_zscore(rep3_df, times_zscore=3)

/tmp/ipykernel_2908/3382132649.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  return df[m], barcodes_removed
/tmp/ipykernel_2908/3382132649.py:10: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  return df[m], barcodes_removed


In [12]:
# set replicate
rep1_df_outliered['replicate'] = "1"
rep2_df_outliered['replicate'] = "2"
rep3_df_outliered['replicate'] = "3"


In [13]:
# concatenate:
df_outliered = pd.concat([rep1_df_outliered, rep2_df_outliered, rep3_df_outliered])

In [14]:
# only important columns
df_outliered = df_outliered[['Barcode', 'name', 'dna_count', 'rna_count', 'replicate']]

In [ ]:
print(df_outliered['name'].nunique()) # official strand sensitive: 75582; bwa finest: 77303; WTC11: 76775
df_outliered.head()

76775


,Barcode,name,dna_count,rna_count,replicate
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,13.0,1
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,9.0,3.0,1
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,10.0,15.0,1
7,CAGTAAACGCCACCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,8.0,1
10,GCGCTAAGGAGGACA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,8.0,13.0,1


In [16]:
df_outliered_filtered = df_outliered.groupby(["name", "replicate"]).filter(lambda x: len(x) >= bc_thresh)

In [17]:
print(df_outliered_filtered['name'].nunique()) # selfmade strand sensitive: 73288; official strand sensitive: 71902; bwa_finest: 73447
df_outliered_filtered.head()

69342


,Barcode,name,dna_count,rna_count,replicate
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,13.0,1
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,9.0,3.0,1
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,10.0,15.0,1
7,CAGTAAACGCCACCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,8.0,1
10,GCGCTAAGGAGGACA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,8.0,13.0,1


In [18]:
# """
# Merge the associated barcode count files of all replicates.
# """

def pivot_table(df, replicates):
    # pivot table to make a dna and rna count column for every replicate
    df = df.pivot_table(
        values=["dna_count", "rna_count"],
        index=["Barcode", "name"],
        columns="replicate",
        aggfunc='first'
    )
    df = df.sort_values("name")


    # order columns to have dna then rna count of each replicate
    col_order = sum(
        [
            ["dna_count_" + rep, "rna_count_" + rep]
            for rep in replicates
        ],
        [],
    )

    df = df.reset_index()

    df.columns = ['_'.join(col).strip() if col[1] else col[0] for col in df.columns.values]

    df = df[["Barcode", "name"] + col_order]

    for col in col_order:
        df[col] = df[col].astype('Int32')

    return df

In [19]:
replicates = ["1","2","3"]
pivot_table_outlier_filter = pivot_table(df_outliered_filtered,replicates)

In [ ]:
print(pivot_table_outlier_filter['name'].nunique()) # selfmade strand sensitive: 73288; official strand sensitive: 71902; bwa_finest: 73447; WTC11: 69342
pivot_table_outlier_filter.head()

69342


,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,GTGGTGTGTGACACC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,8,20,<NA>,<NA>,3,10
1,TCAACACGGCCTCAA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2,13,1,6,<NA>,<NA>
2,GAATGTAACGCCTTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,<NA>,<NA>,<NA>,<NA>,1,4
3,CCAAGTCAGCGAGGT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2,7,5,4,<NA>,<NA>
4,GTCTGGGGAGCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,<NA>,<NA>,2,9,<NA>,<NA>


In [ ]:
pivot_table_outlier_filter.shape[0] # official strand sensitive: 6256755

In [21]:
# write to output file
pivot_table_outlier_filter.to_csv(output_path_outliered, sep="\t", index=False, compression="gzip") # official strand sensitive: 6256755

### Variant specific
- explode the name column which include "#" because of same sequence

In [24]:
columns_name = 'name'
BCalm_variant_input_df_expanded = pivot_table_outlier_filter.assign(name=pivot_table_outlier_filter[columns_name].str.split("#")).explode(columns_name) # official strand sensitive: 6390450; bwa finest: 6486053

In [25]:
BCalm_variant_input_df_expanded.shape[0]

4946364

In [43]:
BCalm_variant_input_df_expanded.to_csv(output_path_split_hashtag, sep="\t", index=False, compression="gzip")
# takes 2min

In [ ]:
BCalm_variant_input_df_expanded.shape[0]
BCalm_variant_input_df_expanded['name'].nunique() # selfmade strand specific: 73847, official strand specific: 72451, bwa_finest: 74009 WTC11 strand sensitive: 69342

69342

In [28]:
BCalm_variant_input_df_expanded.head()

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,GTGGTGTGTGACACC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,8,20,<NA>,<NA>,3,10
1,TCAACACGGCCTCAA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2,13,1,6,<NA>,<NA>
2,GAATGTAACGCCTTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,<NA>,<NA>,<NA>,<NA>,1,4
3,CCAAGTCAGCGAGGT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2,7,5,4,<NA>,<NA>
4,GTCTGGGGAGCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,<NA>,<NA>,2,9,<NA>,<NA>


In [29]:
label_file = BCalm_variant_input_df_expanded['name'].unique()

label_file_df = pd.DataFrame(label_file, columns=['name'])

In [30]:
label_file_df['label'] = label_file_df['name'].apply(hf.get_label)

In [31]:
label_file_df['label'].unique()

array(['C_SLEA', 'C_negative_heart_MK', 'C_negative_neuron_MK',
       'C_negative_neuron_NP', 'C_positive_heart_AB',
       'C_positive_heart_CAD', 'C_positive_heart_MK',
       'C_positive_neuron_CD', 'C_positive_neuron_MK',
       'C_positive_neuron_NP', 'GC_Atrial_fib', 'GC_Cort_Chengyu',
       'GC_DNase_negative_blood', 'GC_DNase_negative_blood_shuffeled',
       'GC_DNase_negative_brain', 'GC_DNase_negative_brain_shuffeled',
       'GC_DNase_positive', 'GC_DNase_positive_shuffeled',
       'GC_GABA_Chengyu', 'GC_Glut_Chengyu', 'GC_Hon', 'GC_Kircher',
       'GC_Liang', 'GC_Mendelian_variants', 'GC_Mohlke', 'GC_Selvarajan',
       'GC_Vista', 'MK', 'cardiac_neuro_cava_random'], dtype=object)

In [44]:
output_file_label = f"/home/kisa/coding/80K_MPRA/design_data/design_info/{cell_type_name}2025_outlier_filtered_label_{group_name_prefix}.tsv"

label_file_df.to_csv(output_file_label, sep="\t", index=False)

In [45]:
output_file_label

'/home/kisa/coding/80K_MPRA/design_data/design_info/WTC11_2025_outlier_filtered_label_strand_sensitive_.tsv'

In [33]:
BCalm_variant_input_df_expanded['name'].isna().sum()

0

In [ ]:
BCalm_variant_input_df_expanded['name'].nunique() # selfmade strand specific: 73847, official strand specific: 72451, bwa_finest: 74009, WTC11 strand sensitive: 69342
label_file_df['name'].nunique() # selfmade strand specific: 73847, official strand specific: 72451, bwa_finest: 74009, WTC11 strand sensitive: 69342

69342

### Investigate the result of the outlier removal
- expanded and with outlier filter: (n_oligos) selfmade strand specific: 73847, official strand specific: 72451
- expanded but no outlier removal: 
    - number of barcodes: selfmade strand specific: 6499042, official strand sensitive: 6442150, bwa_finest:  6539175,
    - number of oligos: official strand sensitive: 76866, bwa_finest: 78571

In [36]:
columns_name = 'name'
BCalm_variant_input_df_expanded_no_outlier_removal = BCalm_variant_input_df.assign(name=BCalm_variant_input_df[columns_name].str.split("#")).explode(columns_name)



In [ ]:
BCalm_variant_input_df_expanded_no_outlier_removal # selfmade strand specific: 6499042, official strand sensitive: 6442150, bwa_finest: 6539175, WTC11 strand sensitive: 5,031,012

,Barcode,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,GCGGTTACTTCTAGC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,3.0,13.0,4.0,13.0,NaN,NaN
1,TGCGGCGCCCCCGCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,9.0,3.0,1.0,15.0,2.0,21.0
2,GAGCACGACAACAGA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,5.0,11.0
3,CAGAGTAGCCATGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,10.0,15.0,2.0,19.0,NaN,NaN
4,CTCAGCGCCCCAATT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,2.0,5.0,NaN,NaN
...,...,...,...,...,...,...,...,...
5031007,GATGCCAATATATAT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,1.0,1.0,NaN,NaN,NaN,NaN
5031008,TTGCCAAGCGGTACT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,9.0,4.0,5.0,21.0
5031009,CGTAAATGTATGCGT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,1.0,1.0,3.0,1.0
5031010,GGGATGATTTCTGGG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,1.0,8.0,2.0,3.0,11.0,14.0


In [ ]:
BCalm_variant_input_df_expanded_no_outlier_removal['name'].nunique() # official strand: 76000, bwa_finest: 78571 WTC11 strand sensitive: 77715

77715

In [42]:
print(f"Fraction of lost oligos because of outlier removal: {pivot_table_outlier_filter['name'].nunique()/BCalm_variant_input_df_expanded_no_outlier_removal['name'].nunique()}")

Fraction of lost oligos because of outlier removal: 0.8922601814321559


In [41]:
output_path_split_hashtag

'/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/strand_sensitive_WTC11_allreps_merged_barcode_assigned_counts_hashtag_expanded.tsv.gz'

In [ ]:
# BCalm_variant_input_df_expanded_no_outlier_removal.to_csv(output_path_split_hashtag, sep="\t", index=False, compression="gzip")


#### Investigate the difference between selfmade strand specific and official strand specific in the oligos measured

In [44]:
# official strand sensitive
official_strand_oligo_set = set(BCalm_variant_input_df_expanded.name.to_list())

In [42]:
input_selfmade_strand = "/home/kisa/coding/80K_MPRA/server_results/80k_counts_after_metadatafile/NGN2_allreps_merged_barcode_assigned_counts_hashtag_expanded.tsv.gz"
selfmade_strand_expanded_outlier_removed = pd.read_csv(input_selfmade_strand, sep="\t")

In [46]:
selfmade_strand_expanded_outlier_removed.name.nunique()
selfmade_oligo_set = set(selfmade_strand_expanded_outlier_removed.name.to_list())

In [ ]:
only_official_strand_oligos = official_strand_oligo_set - selfmade_oligo_set
print(f"Number of oligos only in official strand results: {len(only_official_strand_oligos)}") #31
only_selfmade_strand_oligos = selfmade_oligo_set - official_strand_oligo_set
print(f"Number of oligos only in selfmade strand results: {len(only_selfmade_strand_oligos)}") #1427


Number of oligos only in official strand results: 31
Number of oligos only in selfmade strand results: 1427


In [54]:
only_selfmade_oligos_df = pd.DataFrame(data=only_selfmade_strand_oligos, columns=['name'])
only_selfmade_oligos_df['label'] = only_selfmade_oligos_df['name'].apply(hf.get_label)
only_selfmade_oligos_df['label'].value_counts()

label
cardiac_neuro_cava_random    1382
MK                             14
GC_Glut_Chengyu                12
GC_GABA_Chengyu                 9
GC_Mendelian_variants           4
GC_Mohlke                       2
GC_Kircher                      2
C_positive_neuron_NP            1
C_negative_neuron_MK            1
Name: count, dtype: int64

In [55]:
only_official_oligos_df = pd.DataFrame(data=only_official_strand_oligos, columns=['name'])
only_official_oligos_df['label'] = only_official_oligos_df['name'].apply(hf.get_label)
only_official_oligos_df['label'].value_counts()

label
cardiac_neuro_cava_random    25
GC_Mendelian_variants         2
C_positive_neuron_MK          1
GC_Mohlke                     1
C_SLEA                        1
C_positive_heart_AB           1
Name: count, dtype: int64

### Label file: 
- adding scrambled group label 

In [47]:
def add_scramble_label(row, name_col='name', label_col='label'):
    """Iterate over name and identify headers which are scrambled sequences and add the scramble label"""
    sequence_name = row[name_col]
    if "scramb" in sequence_name:
        row[label_col] = "C_scrambled_sequences_NP_MK"
    return row

In [48]:
labelfile = "/home/kisa/coding/80K_MPRA/design_data/design_info/2025_outlier_filtered_label.tsv"
labelfile = output_file_label # '/home/kisa/coding/80K_MPRA/design_data/design_info/WTC11_2025_outlier_filtered_label_strand_sensitive_.tsv'
label_df =  pd.read_csv(labelfile, sep="\t")

In [49]:
label_df

,name,label
0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,C_SLEA
1,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,C_SLEA
2,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,C_SLEA
3,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,C_SLEA
4,C_SLEA:SLEA_hg18:chr2:210861483-210861650|10:V...,C_SLEA
...,...,...
69337,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random
69338,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random
69339,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random
69340,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,cardiac_neuro_cava_random


In [50]:
# find scramble sequences:
scrambled_sequences = label_df.loc[label_df['name'].str.contains('scramb')]

In [51]:
mod_label_df = label_df.apply(add_scramble_label,axis=1).copy()

In [52]:
mod_label_df["label"].value_counts()

label
cardiac_neuro_cava_random            64285
MK                                    1539
C_positive_heart_AB                    583
C_scrambled_sequences_NP_MK            469
GC_Selvarajan                          312
GC_Vista                               222
GC_Mendelian_variants                  197
C_negative_heart_MK                    185
GC_Cort_Chengyu                        185
GC_Kircher                             177
C_SLEA                                 175
C_negative_neuron_MK                   167
C_negative_neuron_NP                   100
C_positive_neuron_MK                    94
C_positive_heart_CAD                    90
GC_Glut_Chengyu                         82
C_positive_heart_MK                     81
C_positive_neuron_NP                    62
C_positive_neuron_CD                    61
GC_DNase_positive_shuffeled             51
GC_Atrial_fib                           44
GC_DNase_positive                       37
GC_GABA_Chengyu                         35
GC_Mo

In [53]:
output_scrambled_label_file = f"/home/kisa/coding/80K_MPRA/design_data/design_info/{cell_type_name}2025_outlier_filtered_label_with_scrambled.tsv"
mod_label_df.to_csv(output_scrambled_label_file, sep="\t", index=False)

In [54]:
output_scrambled_label_file

'/home/kisa/coding/80K_MPRA/design_data/design_info/WTC11_2025_outlier_filtered_label_with_scrambled.tsv'

### Harmonize the sequence names: 
- for WTC11 MPRAsnakeflow results a different design file was used and need to have same headers => replace "~" with ";" and * with ">" and add (.) to the headers of these groups: ">GC_DNase_positive:", ">GC_DNase_negative_brain:", ">GC_DNase_negative_blood:"
- merged multiple same sequences with a "#" => if "#" in the header split and only take the one which does not have "_copy" (for MK)
- GC_GABA_Chengyu:GABA|chr1:162285544-162285813|+|1.64#GC_Glut_Chengyu:Glut|chr1:162285544-162285813|+|1.58
  - replace "#" with ";"
- C_negative_heart_MK:tile_12290_chr14_68731029_68731298_reference__0.278829455387256#C_negative_neuron_MK:tile_12290_chr14_68731029_68731298_reference__0.278829455387256
  - replace "#" with ";"
- GC_GABA_Chengyu:GABA|chr12:29884985-29885254|+|1.82#MK:tile_9261|chr12-29884985+29885254|reference
  - replace "#" with ";"

In [33]:
import pandas as pd

In [19]:
# one code block for ngn2
# load the barcode results
mprasnakeflow_barcode_results_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/experiment/20241113_80K_MPRAsnakeflow/results/experiments/mpra80KNeuronbbmapmapq30BC10DNA1RNA1SelfmadeStrandsensitivity/assigned_counts/assignmentFixDuplicates/default/NGN2_allreps_merged_barcode_assigned_counts.tsv.gz"
mprasnakeflow_barcode_results = pd.read_csv(mprasnakeflow_barcode_results_path, sep="\t")

# read the used design file in
used_design_file = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed_replaced_comma_with_tilde_greater_to_star_no_brackets.fa"
# NOTE: sequence without adapter needed for the left join
used_design = hf.fasta_to_dataframe(used_design_file, columns=["NGN2_name", "sequence_with_adapter"])
def remove_adapter(sequence):
    return sequence[15:-15]
used_design['sequence'] = used_design['sequence_with_adapter'].apply(remove_adapter)


# read metadata and merge
metadata_file_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/MPRA_80215_April_server.tsv.gz"
metadata_file_df = pd.read_csv(metadata_file_path, sep="\t", low_memory=True)

import ast

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# use safe eval on list
# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_file_df[col] = metadata_file_df[col].apply(safe_eval)


metadata_merged = metadata_file_df[['name', col_sequence]].merge(used_design, on=col_sequence, how="inner")


In [20]:
used_design

,NGN2_name,sequence_with_adapter,sequence
0,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTAAGAATACAAGTAACTGATGAATGAAGGGGG...,AAGAATACAAGTAACTGATGAATGAAGGGGGCATCTTGTGTCCCCA...
1,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTTTGGGTATGCTGCCCCCCAGCTGGCGGGGCA...,TTGGGTATGCTGCCCCCCAGCTGGCGGGGCACCGGGGACAGGCACA...
2,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTACGAGCAAGGGAATGAGAGAGAGTGGGTTAG...,ACGAGCAAGGGAATGAGAGAGAGTGGGTTAGAGAGTGAGTGAGCCA...
3,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCGTGGACACGCGTGATTGACCCTTTAACTGT...,CGTGGACACGCGTGATTGACCCTTTAACTGTATCCTTAACCACCGC...
4,cardiac_neuro_cava_random:SKI|ENSG00000157933....,AGGACCGGATCAACTCCGGAGAGTCTCAGCTCCCGCAGCCCTAACA...,CCGGAGAGTCTCAGCTCCCGCAGCCCTAACAAACGACCACAGACCT...
...,...,...,...
80210,MK:tile_2240|chr1-116244322+116244591|scramble...,AGGACCGGATCAACTCTTAATCAAATAACCCATTAATTCTATATAT...,CTTAATCAAATAACCCATTAATTCTATATATCTACCTAATATTAAT...
80211,MK:tile_6675|chr11-2374617+2374886|scramble_ne...,AGGACCGGATCAACTCATCGGCCCTGGTGAAGCGTCCGTCCAGACG...,CATCGGCCCTGGTGAAGCGTCCGTCCAGACGGGCCTGCCTAGCCTC...
80212,MK:tile_18415|chr17-71181691+71181960|scramble...,AGGACCGGATCAACTTAAATATTCAGCGATACATTCCTATTCTTTT...,TAAATATTCAGCGATACATTCCTATTCTTTTTCAGAAGTAGTTATT...
80213,MK:tile_14356|chr15-67031618+67031887|scramble...,AGGACCGGATCAACTTGAAGCCCCTGATTCTGTTAGAATAAGGTTA...,TGAAGCCCCTGATTCTGTTAGAATAAGGTTACTGAGTCGTGTATAC...


In [21]:
metadata_file_df

,name,sequence,category,class,source,ref,chr,start,end,strand,variant_class,variant_pos,SPDI,allele,info
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,159752058.0,159752328.0,+,[SNV],[234],[NC_000001.11:159752292:A:G],[ref],NaN
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159023.0,230159293.0,+,[SNV],[145],[NC_000001.11:230159168:C:T],[ref],NaN
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136.0,230159406.0,+,"[SNV, indel]","[32, 192]","[NC_000001.11:230159168:C:T, NC_000001.11:2301...","[ref, ref]",NaN
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230159136.0,230159406.0,+,[indel],[192],[NC_000001.11:230159328:TCTTAAAGTGTTCAGCACTCCC:T],[alt],NaN
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,variant,variant negative control,general controls IGVF year 1 design 2023,GRCh38,chr1,230161269.0,230161539.0,+,[SNV],[120],[NC_000001.11:230161389:C:T],[ref],NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,element,element inactive control,NaN,GRCh38,chr12,44940898.0,44941168.0,-,None,None,None,None,NaN
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,element,element inactive control,NaN,GRCh38,chr2,212699380.0,212699650.0,+,None,None,None,None,NaN
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,element,element inactive control,NaN,GRCh38,chr10,26798828.0,26799098.0,-,None,None,None,None,NaN
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,element,element inactive control,NaN,GRCh38,chr7,31260963.0,31261233.0,+,None,None,None,None,NaN


In [22]:
metadata_merged

,name,sequence,NGN2_name,sequence_with_adapter
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,AGGACCGGATCAACTATACATCCTTTAATTTGTTCCTACATCTTGC...
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGGACCGGATCAACTTGTGTCTGGTGAGGTTGCTGACACTGCTTTT...
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGGACCGGATCAACTGTTTACCCAGCCGTGGGAAAGGACGCTGTAC...
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,AGGACCGGATCAACTGTTTACCCAGCCGTGGGAAAGGACGCTGTAC...
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,AGGACCGGATCAACTCCTCAACTCTCCACATGCCCCAGTAGCATAG...
...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,GC_GABA_Chengyu:GABA|chr12:44940899-44941168|-...,AGGACCGGATCAACTTGTCCAAAAAAAGTAAAAGTCATAACAGAAA...
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,GC_GABA_Chengyu:GABA|chr2:212699381-212699650|...,AGGACCGGATCAACTTGTGAGTGCTATAATTGTAACCCTTTATAAT...
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,GC_GABA_Chengyu:GABA|chr10:26798829-26799098|-...,AGGACCGGATCAACTTTTAATAGGAGTACTATTGAATTTACATTTA...
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,GC_GABA_Chengyu:GABA|chr7:31260964-31261233|+|...,AGGACCGGATCAACTTTTCTTATACTGTATGTTTAAAGATATAGAC...


In [23]:
# combine with the name matching table
mprasnakeflow_barcode_results_merged = mprasnakeflow_barcode_results.merge(metadata_merged, left_on='oligo_name', right_on="NGN2_name", how="inner").copy()
# rename the columns
mprasnakeflow_barcode_results_merged['old_oligo_name'] = mprasnakeflow_barcode_results_merged['oligo_name']

mprasnakeflow_barcode_results_merged['oligo_name'] = mprasnakeflow_barcode_results_merged['name']

In [24]:
mprasnakeflow_barcode_results_merged

,barcode,oligo_name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,name,sequence,NGN2_name,sequence_with_adapter,old_oligo_name
0,CTAACGTGAACTGAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,11.0,31.0,4.0,23.0,7.0,30.0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
1,GGTACGACGCACCAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,2.0,7.0,2.0,13.0,4.0,8.0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
2,CCTATCAGTCTCTAT,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,3.0,6.0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
3,ACAACGACTGTCAAC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,5.0,9.0,NaN,NaN,2.0,9.0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
4,CAGATCGGTTGATTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,5.0,10.0,6.0,5.0,2.0,11.0,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,TAGGCTTCTCAAAAGTTATTTTTAAAGACTGAGGAATTAGGCACCT...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,AGGACCGGATCAACTTAGGCTTCTCAAAAGTTATTTTTAAAGACTG...,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
6386968,GCTCCCTTAGTGGGG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,1.0,7.0,2.0,3.0,NaN,NaN,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,GGTTGGGAACTTCTTCGTCTACCTTTCCATTTTCAGATTTGGCACT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,AGGACCGGATCAACTGGTTGGGAACTTCTTCGTCTACCTTTCCATT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...
6386969,GTGGGCGAGCTGAAC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,2.0,1.0,NaN,NaN,NaN,NaN,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,GGTTGGGAACTTCTTCGTCTACCTTTCCATTTTCAGATTTGGCACT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,AGGACCGGATCAACTGGTTGGGAACTTCTTCGTCTACCTTTCCATT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...
6386970,CTCCTCGACTCCGCA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,3.0,2.0,5.0,9.0,4.0,4.0,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,GGTTGGGAACTTCTTCGTCTACCTTTCCATTTTCAGATTTGGCACT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,AGGACCGGATCAACTGGTTGGGAACTTCTTCGTCTACCTTTCCATT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...
6386971,TTGAGGCAGCTAGAT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,1.0,5.0,NaN,NaN,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,GGTTGGGAACTTCTTCGTCTACCTTTCCATTTTCAGATTTGGCACT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,AGGACCGGATCAACTGGTTGGGAACTTCTTCGTCTACCTTTCCATT...,cardiac_neuro_cava_random:ZNF462|ENSG000001481...


In [26]:
writing=False
if writing:
    mprasnakeflow_output_path = "/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/processing_mpralib/80K_reporter_experiment.barcode.NGN2.SelfmadeStrandSensitiveAssignment.default.all.tsv.gz"
    mprasnakeflow_barcode_results_merged[['barcode', 'oligo_name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3']].to_csv(mprasnakeflow_output_path, sep="\t", index=False)

##### Same for the WTC11 data

In [34]:
wtc11_mprasnakeflow_barcode_results = "/data/cephfs-2/unmirrored/groups/kircher/IGVF_data_submssion_TM/80K/mprasnakeflow/results/experiments/defaultWTC11Resequencing/reporter_experiment.barcode.WTC11.defaultAssignmentBBMap.default.all.tsv.gz"
# wtc11_mprasnakeflow_barcode_results = "/home/kisa/coding/80K_MPRA/80K-Analysis/06_variant_element_analysis/scripts/testing_reading_bc_data.tsv"
wtc11_mprasnakeflow_results = pd.read_csv(wtc11_mprasnakeflow_barcode_results, sep="\t")


In [35]:
wtc11_mprasnakeflow_results.columns

Index(['barcode', 'oligo_name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3'],
      dtype='object')

In [ ]:
wtc11_mprasnakeflow_results

In [ ]:
design_file_path = "/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/resources/association_data/design_removed_spaces_deduplicated_sequences_renamed_replaced_comma_with_tilde_greater_to_star_no_brackets.fa"
design_file_df = hf.fasta_to_dataframe(design_file_path, columns=["wtc11_name", "sequence_with_adapter"])
design_file_df

# rename sequence column
# design_file_df = design_file_df.rename(columns={"sequence": "sequence_with_adapter"}).copy()

def remove_adapter(sequence):
    return sequence[15:-15]

design_file_df['sequence'] = design_file_df['sequence_with_adapter'].apply(remove_adapter)

In [40]:
metadata_file_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/MPRA_80215_April_server.tsv.gz"
metadata_file_df = pd.read_csv(metadata_file_path, sep="\t", low_memory=True)

import ast

# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'

list_columns = [col_variant_class, col_variant_pos, col_SPDI, col_allele]

# Function to safely evaluate string representations of lists
def safe_eval(x):
    if pd.isna(x):
        return None
    try:
        return ast.literal_eval(x)
    except (ValueError, SyntaxError):
        return x

# use safe eval on list
# Apply the safe_eval function to the specified columns
for col in list_columns:
    metadata_file_df[col] = metadata_file_df[col].apply(safe_eval)


metadata_merged = metadata_file_df[['name', col_sequence]].merge(design_file_df, on=col_sequence, how="inner")

In [41]:
metadata_merged

,name,sequence,wtc11_name,sequence_with_adapter
0,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,ATACATCCTTTAATTTGTTCCTACATCTTGCTTGGATTTTCCCCTG...,GC_Mohlke:REF_NC000001.11|159752292|A|G|Mohlke...,AGGACCGGATCAACTATACATCCTTTAATTTGTTCCTACATCTTGC...
1,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,TGTGTCTGGTGAGGTTGCTGACACTGCTTTTGGATGAGAGAGAGAG...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGGACCGGATCAACTTGTGTCTGGTGAGGTTGCTGACACTGCTTTT...
2,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,GC_Mohlke:REF_NC000001.11|230158967|C|A|Mohlke...,AGGACCGGATCAACTGTTTACCCAGCCGTGGGAAAGGACGCTGTAC...
3,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,GTTTACCCAGCCGTGGGAAAGGACGCTGTACCCCTGCCCTATTGGC...,GC_Mohlke:ALT_NC000001.11|230158967|C|A|Mohlke...,AGGACCGGATCAACTGTTTACCCAGCCGTGGGAAAGGACGCTGTAC...
4,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,CCTCAACTCTCCACATGCCCCAGTAGCATAGACCAGCTTCCTTACA...,GC_Mohlke:REF_NC000001.11|230161389|C|T|Mohlke...,AGGACCGGATCAACTCCTCAACTCTCCACATGCCCCAGTAGCATAG...
...,...,...,...,...
80210,GC_Glut_Chengyu:Glut|chr12:44940899-44941168|-...,TGTCCAAAAAAAGTAAAAGTCATAACAGAAATTGGATTTCAAATGG...,GC_GABA_Chengyu:GABA|chr12:44940899-44941168|-...,AGGACCGGATCAACTTGTCCAAAAAAAGTAAAAGTCATAACAGAAA...
80211,GC_Glut_Chengyu:Glut|chr2:212699381-212699650|...,TGTGAGTGCTATAATTGTAACCCTTTATAATTGACAGACATTAATT...,GC_GABA_Chengyu:GABA|chr2:212699381-212699650|...,AGGACCGGATCAACTTGTGAGTGCTATAATTGTAACCCTTTATAAT...
80212,GC_Glut_Chengyu:Glut|chr10:26798829-26799098|-...,TTTAATAGGAGTACTATTGAATTTACATTTAATGTAGTTAATGATA...,GC_GABA_Chengyu:GABA|chr10:26798829-26799098|-...,AGGACCGGATCAACTTTTAATAGGAGTACTATTGAATTTACATTTA...
80213,GC_Glut_Chengyu:Glut|chr7:31260964-31261233|+|...,TTTCTTATACTGTATGTTTAAAGATATAGACAATGCATATACAAGT...,GC_GABA_Chengyu:GABA|chr7:31260964-31261233|+|...,AGGACCGGATCAACTTTTCTTATACTGTATGTTTAAAGATATAGAC...


In [42]:
wtc11_mprasnakeflow_results

,barcode,oligo_name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,TGTATGCCTTATCCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,5.0,19.0,5.0,8.0
1,GCGTTACAGTCCTTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,4.0,19.0,NaN,NaN,NaN,NaN
2,CAATGGAAGCCAGAC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,8.0,29.0,NaN,NaN,11.0,27.0
3,AGCGGAATTTGTCTA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,6.0,5.0,NaN,NaN,NaN,NaN
4,CTCGTTCACCCTTAG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,8.0,8.0
...,...,...,...,...,...,...,...,...
5533293,ATCTCCACCAGGAAT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,NaN,NaN,6.0,4.0
5533294,TTTGATGTCTTGACG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,7.0,10.0,1.0,18.0,15.0,23.0
5533295,GCCAACTTCCCCTGT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,4.0,26.0,5.0,11.0
5533296,TACCCACTTAAGGCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,6.0,22.0,4.0,37.0,5.0,16.0


In [ ]:
wtc11_mprasnakeflow_results

# combine with the name matching table
wtc11_mprasnakeflow_results_merged = wtc11_mprasnakeflow_results.merge(metadata_merged, left_on='oligo_name', right_on="wtc11_name", how="inner").copy()
# rename the columns
wtc11_mprasnakeflow_results_merged['old_oligo_name'] = wtc11_mprasnakeflow_results_merged['oligo_name']

wtc11_mprasnakeflow_results_merged['oligo_name'] = wtc11_mprasnakeflow_results_merged['name']


In [ ]:
# write the file
# ['barcode', 'oligo_name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
#        'rna_count_2', 'dna_count_3', 'rna_count_3']
writing=False
if writing:
    wtc11_output_path = "/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/processing_mpralib/80K_reporter_experiment.barcode.WTC11.defaultAssignmentBBMap.default.all.tsv.gz"
    wtc11_mprasnakeflow_results_merged[['barcode', 'oligo_name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3']].to_csv(wtc11_output_path, sep="\t", index=False)

### Optional: differences between cell-types
- goal: input for BCalm for the element wise cell-type difference table

In [53]:
wtc11_barcode_path = "/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/processing_mpralib/80k_WTC11_resequencing_normalized_counts.tsv"

In [54]:
wtc11_barcode_data = pd.read_csv(wtc11_barcode_path, sep="\t")
wtc11_barcode_data

,ID,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,TGTATGCCTTATCCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,0.165490,0.250416,0.178691,0.104983
1,GCGTTACAGTCCTTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,0.141862,0.274673,NaN,NaN,NaN,NaN
2,CAATGGAAGCCAGAC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,0.255352,0.412009,NaN,NaN,0.357382,0.326614
3,AGCGGAATTTGTCTA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,0.198607,0.082402,NaN,NaN,NaN,NaN
4,CTCGTTCACCCTTAG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,0.268036,0.104983
...,...,...,...,...,...,...,...,...
5533293,ATCTCCACCAGGAAT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,NaN,NaN,0.208473,0.058324
5533294,TTTGATGTCTTGACG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.226980,0.151070,0.055163,0.237896,0.476509,0.279955
5533295,GCCAACTTCCCCTGT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,0.137908,0.338062,0.178691,0.139977
5533296,TACCCACTTAAGGCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.198607,0.315874,0.137908,0.475791,0.178691,0.198301


In [57]:
wtc11_barcode_data['cell_type'] = "undiff_WTC11"

In [58]:
wtc11_barcode_data

,ID,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,cell_type
0,TGTATGCCTTATCCG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,0.165490,0.250416,0.178691,0.104983,undiff_WTC11
1,GCGTTACAGTCCTTC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,0.141862,0.274673,NaN,NaN,NaN,NaN,undiff_WTC11
2,CAATGGAAGCCAGAC,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,0.255352,0.412009,NaN,NaN,0.357382,0.326614,undiff_WTC11
3,AGCGGAATTTGTCTA,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,0.198607,0.082402,NaN,NaN,NaN,NaN,undiff_WTC11
4,CTCGTTCACCCTTAG,C_SLEA:SLEA_hg18:chr2:210861483-210861650|103:...,NaN,NaN,NaN,NaN,0.268036,0.104983,undiff_WTC11
...,...,...,...,...,...,...,...,...,...
5533293,ATCTCCACCAGGAAT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,NaN,NaN,0.208473,0.058324,undiff_WTC11
5533294,TTTGATGTCTTGACG,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.226980,0.151070,0.055163,0.237896,0.476509,0.279955,undiff_WTC11
5533295,GCCAACTTCCCCTGT,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,0.137908,0.338062,0.178691,0.139977,undiff_WTC11
5533296,TACCCACTTAAGGCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.198607,0.315874,0.137908,0.475791,0.178691,0.198301,undiff_WTC11


In [48]:
ngn2_barcode_path = "/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/processing_mpralib/80k_NGN2_normalized_counts.tsv.gz"

In [49]:
ngn2_barcode_data = pd.read_csv(ngn2_barcode_path, sep="\t")
ngn2_barcode_data

,ID,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3
0,TTGTTAAAGGGGAGG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.078556,0.134817,0.153148,0.032023,0.113516,0.046527
1,GTTCGTCTAAGTTTG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.549889,0.314573,0.191435,0.368259,0.151355,0.511797
2,GACGCCCAGCCGCTG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.117833,0.134817,0.114861,0.048034,0.075677,0.108563
3,TGTCGTCTCCGCTCG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.078556,0.119837,0.153148,0.192135,0.189194,0.139581
4,TCGCGCATCACGACG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.196389,0.224695,0.076574,0.144102,0.264871,0.155090
...,...,...,...,...,...,...,...,...
6370543,GGGGATGCGGTAGGA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.314222,0.209716,0.268009,0.288203,0.264871,0.170599
6370544,ATGTAAAGACCCCCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.196389,0.314573,0.306296,0.400282,0.302710,0.294671
6370545,GATCGACGTTTTTAA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.078556,0.224695,0.229722,0.256180,0.189194,0.155090
6370546,ACTATGTTATCCAAC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,0.191435,0.176124,0.113516,0.093054


In [ ]:
ngn2_barcode_data['cell_type'] = "NGN2_neurons"

In [56]:
ngn2_barcode_data

,ID,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,cell_type
0,TTGTTAAAGGGGAGG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.078556,0.134817,0.153148,0.032023,0.113516,0.046527,NGN2_neurons
1,GTTCGTCTAAGTTTG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.549889,0.314573,0.191435,0.368259,0.151355,0.511797,NGN2_neurons
2,GACGCCCAGCCGCTG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.117833,0.134817,0.114861,0.048034,0.075677,0.108563,NGN2_neurons
3,TGTCGTCTCCGCTCG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.078556,0.119837,0.153148,0.192135,0.189194,0.139581,NGN2_neurons
4,TCGCGCATCACGACG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.196389,0.224695,0.076574,0.144102,0.264871,0.155090,NGN2_neurons
...,...,...,...,...,...,...,...,...,...
6370543,GGGGATGCGGTAGGA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.314222,0.209716,0.268009,0.288203,0.264871,0.170599,NGN2_neurons
6370544,ATGTAAAGACCCCCC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.196389,0.314573,0.306296,0.400282,0.302710,0.294671,NGN2_neurons
6370545,GATCGACGTTTTTAA,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,0.078556,0.224695,0.229722,0.256180,0.189194,0.155090,NGN2_neurons
6370546,ACTATGTTATCCAAC,cardiac_neuro_cava_random:ZNF462|ENSG000001481...,NaN,NaN,0.191435,0.176124,0.113516,0.093054,NGN2_neurons


In [59]:
# concat the barcode data
combined_barcode_data = pd.concat([ngn2_barcode_data, wtc11_barcode_data], ignore_index=True)

In [ ]:
combined_barcode_data['cell_type'].value_counts()
# NGN2_neurons    6370548
# undiff_WTC11    5533298

cell_type
NGN2_neurons    6370548
undiff_WTC11    5533298
Name: count, dtype: int64

In [ ]:
combined_barcode_data

In [63]:
combined_barcode_data['name_with_cell_type'] = combined_barcode_data["name"] + "_" + combined_barcode_data['cell_type']

In [72]:
combined_barcode_data['name_without_cell_type'] = combined_barcode_data['name']
combined_barcode_data['name'] = combined_barcode_data['name_with_cell_type']
combined_barcode_data['barcode'] = combined_barcode_data['ID']

In [73]:
combined_barcode_data.head()

,ID,name,dna_count_1,rna_count_1,dna_count_2,rna_count_2,dna_count_3,rna_count_3,cell_type,name_with_cell_type,name_without_cell_type,barcode
0,TTGTTAAAGGGGAGG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.078556,0.134817,0.153148,0.032023,0.113516,0.046527,NGN2_neurons,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...,TTGTTAAAGGGGAGG
1,GTTCGTCTAAGTTTG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.549889,0.314573,0.191435,0.368259,0.151355,0.511797,NGN2_neurons,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...,GTTCGTCTAAGTTTG
2,GACGCCCAGCCGCTG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.117833,0.134817,0.114861,0.048034,0.075677,0.108563,NGN2_neurons,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...,GACGCCCAGCCGCTG
3,TGTCGTCTCCGCTCG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.078556,0.119837,0.153148,0.192135,0.189194,0.139581,NGN2_neurons,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...,TGTCGTCTCCGCTCG
4,TCGCGCATCACGACG,GC_DNase_negative_blood:chr10:16877280-1687754...,0.196389,0.224695,0.076574,0.144102,0.264871,0.155090,NGN2_neurons,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...,TCGCGCATCACGACG


In [70]:
combined_barcode_data.columns

Index(['ID', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3', 'cell_type',
       'name_with_cell_type'],
      dtype='object')

In [ ]:
# get all different values from the dataframe names

# Get unique names
unique_names = combined_barcode_data['name_without_cell_type'].unique()

# Create the new dataframe
element_match_table = pd.DataFrame({
    'ID': unique_names,
    'REF': [name + "_" + 'NGN2_neurons' for name in unique_names],
    'ALT': [name + "_" +  'undiff_WTC11' for name in unique_names]
})

In [75]:
element_match_table

,ID,REF,ALT
0,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...,GC_DNase_negative_blood:chr10:16877280-1687754...
1,GC_DNase_negative_blood:chr12:4360422-4360691_...,GC_DNase_negative_blood:chr12:4360422-4360691_...,GC_DNase_negative_blood:chr12:4360422-4360691_...
2,GC_DNase_negative_blood:chr14:78017728-7801799...,GC_DNase_negative_blood:chr14:78017728-7801799...,GC_DNase_negative_blood:chr14:78017728-7801799...
3,GC_DNase_negative_blood:chr17:14829903-1483017...,GC_DNase_negative_blood:chr17:14829903-1483017...,GC_DNase_negative_blood:chr17:14829903-1483017...
4,GC_DNase_negative_blood:chr18:29321172-2932144...,GC_DNase_negative_blood:chr18:29321172-2932144...,GC_DNase_negative_blood:chr18:29321172-2932144...
...,...,...,...
77963,cardiac_neuro_cava_random:REF_SMAD3|ENSG000001...,cardiac_neuro_cava_random:REF_SMAD3|ENSG000001...,cardiac_neuro_cava_random:REF_SMAD3|ENSG000001...
77964,cardiac_neuro_cava_random:REF_TBC1D24|ENSG0000...,cardiac_neuro_cava_random:REF_TBC1D24|ENSG0000...,cardiac_neuro_cava_random:REF_TBC1D24|ENSG0000...
77965,cardiac_neuro_cava_random:REF_TBX5|ENSG0000008...,cardiac_neuro_cava_random:REF_TBX5|ENSG0000008...,cardiac_neuro_cava_random:REF_TBX5|ENSG0000008...
77966,cardiac_neuro_cava_random:REF_TMEM115|ENSG0000...,cardiac_neuro_cava_random:REF_TMEM115|ENSG0000...,cardiac_neuro_cava_random:REF_TMEM115|ENSG0000...


In [7]:
element_match_table.to_csv("/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/80k_combined_cell_types_matching_names.tsv.gz", sep="\t", index=False)

In [78]:
# write to file:
writing=False
writing=True
if writing:
    combined_barcode_path = "/home/kisa/coding/80K_MPRA/element_analysis_output/202502_element_analysis/80k_combined_celltype_element_data_2025_05.tsv.gz"
    combined_barcode_data[['barcode', 'name', 'dna_count_1', 'rna_count_1', 'dna_count_2',
       'rna_count_2', 'dna_count_3', 'rna_count_3', 'cell_type',
       'name_without_cell_type']].to_csv(combined_barcode_path, sep="\t", index=False)